## Введение

В этом дз вам предстоит сравнить llm с разными квантизациями и построить небольшой бенчмарк 

Далее будем использовать для развертывании `ollama` - https://ollama.com/

1. Сначала запустим локальный сервер командой `!ollama serve`

2. Можно запустить интерактивную сессию, используя следующую команду: `!ollama run qwen2.5:7b`
Причем модель изначально может не лежать локально: в этом случае ollama сам подтянет ее из своего хаба.

3. Можно скачать модель из хаба ollama командой `!ollama pull qwen2.5:7b`, а затем ее использовать  

4. Также можно самим создать модель, которую вы предварительно скачали из любого места (ollama hub, hugging face hub и тд).
Для этого нужно создать файл `Modelname.txt` (https://ollama.readthedocs.io/en/modelfile/). И запустить следующую команду: `!ollama create qwen2.5-7b-instruct-int2 -f Modelname.txt`

In [168]:
!cat Modelname_1.txt

FROM qwen2.5-7b-instruct-q2_k.gguf
PARAMETER temperature 0.7
PARAMETER num_gpu 0


С помощью `huggingface_hub` можно скачать нужные модельки прямо на сервер. Например:

In [14]:
from huggingface_hub import hf_hub_download
hf_hub_download(repo_id='Qwen/Qwen2.5-7B-Instruct-GGUF', filename='qwen2.5-7b-instruct-q2_k.gguf', local_dir='.')

'qwen2.5-7b-instruct-q2_k.gguf'

1. Можно напрямую отправлять запросы либо использовать python API - https://docs.ollama.com/quickstart
2. Через OpenAI - https://ollama.com/blog/openai-compatibility

Но мы будем использовать удобную обертку из langchain - `ChatOllama` (https://python.langchain.com/docs/integrations/chat/ollama/). Мы подробнее поговорим на следующуем уроке, почему это удобно. Пока просто думаем о `ChatOllama` как о клиенте, который за нас формирует запросы и отправляет из на сервер

In [120]:
from langchain_ollama import ChatOllama

Для построения бенчмарка будем использовать первую часть ЕГЭ по математике, т.е. от модели требуется на выходе только число (несколько чисел).

Мы можем заставить модель выдавать только число на выходе с помощью `structured_output` (https://platform.openai.com/docs/guides/structured-outputs). Для этого можно задать специальную json схему или `pydantic` схему - https://docs.pydantic.dev/latest/ \
С помощью схем мы структурируем выход модели под определенный формат.

## Задания

Подумайте, как должна выглядить схема в случае, если мы хотим, чтобы модель выдала как полноценный ответ на задачу, так и числовой ответ

In [121]:
from typing import List
from pydantic import BaseModel, Field

class MathReasoning(BaseModel):
    pass

Зайдате входной промпт, подумайте, что написать в системном промтпе:

In [166]:
messages = [
 ...   
]

Создадим инстанс модели, указав необходимые параметры

In [133]:
model = 'qwen2.5:7b-instruct-q2_K' # имя скачанной модели или созданной через Modelfile
ollama_model = ChatOllama(
    model=model,
    api_key="ollama",
    base_url="http://localhost:11434/",
    format=MathReasoning.model_json_schema(), # параметр, отвечающий за выход модели (structured output)
    temperature=0,
)

Используем метод `with_structured_output` из langchain, которые парсит выход модели и валидирует через pydantic схему

In [129]:
# проверьте, что все отрабатывается корректно

output = ollama_model.with_structured_output(schema=MathReasoning).invoke(
    messages,
)

Но если вы прокинули `format` при создании модели, вызвать модель можно и через обычный `invoke`, а затем распарсить и провалидировать выход модели

In [144]:
# проверьте, что все отрабатывается корректно

output = ollama_model.invoke(
    messages,
)
parsed = MathReasoning.model_validate(json.loads(output.content))
print(parsed.final_answer) # достаем ответ как атрибут pydantic схемы

Будем проверять модели с разной квантизацией на примере моделей qwen.

Для этого вам надо будет:
1. Скачать три варианта Qwen2.5-7B-Instruct с разными квантизациями: `qwen2.5:7b-instruct-q2_K`, `qwen2.5:7b-instruct-q4_K_M`, `qwen2.5:7b-instruct-q8_0`. Их можно скачать готовыми из [Ollama Library](https://ollama.com/library/qwen2.5/tags) командой `ollama pull` или создать модели из GGUF самостоятельно.

2. Реализовать расчет бенчмарка.\
В бенчмарке хочется увидеть: среднюю скорость генерации (количество токенов в секунду), сколько памяти занимает каждая модель, правильность ответа. \
Сделаем это табличкой: model vs accuracy/metrics

3. Также подумайте, какие параметры генерации нужно зафиксировать (temperature, и тд), чтобы сравнение моделей было честным. Все эти параметры нужно прокинуть при инициализации модели через `ChatOllama`

Реализуйте расчет метрик

In [ ]:
def measure_inference():
    ...
    return {
        "ttft": ,
        "ttl": ,                                   
        "completion_tokens": ,
        'overall_time': ,
        "output":,

        # ITL:
        "itl_ms_avg": ,
    
        # throughput:
        "tokens_per_sec_w/o_prefill": , 
        "tokens_per_sec": 
    }

def get_memory():
    ...
    mem = ... # memory in MB
    return mem

Реализуйте основную функция для прогона моделек и расчета метрик

In [ ]:
MODELS = [] # ваш список моделей
BENCH_DATASET = '' # путь к датасету с бенчмарком (датасет лежит в репозитории)

def compute_benchmark(
    dataset: csv,
    models: List[str],
    output_savepath: Literal[str, None]
):
    """
    Returns: На выходе хочется получить таблицы с метриками. Опционально можно их сохранить в текущую директорию
    """

## Ваши выводы

Опишите кратко ваши результаты: как влияет квантизация на качество, скорость генерации; другие инсайты, если есть